In [ ]:
import time
import multiprocessing as mp
import matplotlib.pyplot as plt
import sys

def cek_prima(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def hitung_prima(rentang):
    mulai, selesai = rentang
    return [n for n in range(mulai, selesai) if cek_prima(n)]

if __name__ == '__main__':
    batas_awal = 1
    batas_akhir = 5
    
    workers_list = [1, 2, 4, 8]
    waktu_eksekusi = {}

    # Membagi rentang kerja menjadi 8 bagian agar bisa diproses parallel
    jumlah_bagian = 8
    chunk = (batas_akhir - batas_awal) // jumlah_bagian
    ranges = [(i * chunk + 1, (i + 1) * chunk if i < 7 else batas_akhir) for i in range(jumlah_bagian)]

    metode_ctx = 'spawn' if sys.platform.startswith('win') else 'fork'
    ctx = mp.get_context(metode_ctx)

    for w in workers_list:
        print(f"Menjalankan dengan {w} worker...")
        start_time = time.time()
        
        # Buat pool di dalam konteks aman
        with ctx.Pool(processes=w) as pool:
            hasil = pool.map(hitung_prima, ranges)
            
        end_time = time.time()
        durasi = end_time - start_time
        waktu_eksekusi[w] = durasi
        print(f"Selesai worker {w}: {durasi:.4f} detik")

    # Hitung Speedup
    t_1 = waktu_eksekusi[1]
    speedup_aktual = [t_1 / waktu_eksekusi[w] for w in workers_list]

    # Hukum Amdahl (P = 0.90)
    P = 0.90 
    speedup_amdahl = [1 / ((1 - P) + (P / w)) for w in workers_list]

    # Plot Grafik
    plt.figure(figsize=(8, 5))
    plt.plot(workers_list, speedup_aktual, marker='o', label='Speedup Aktual', color='blue')
    plt.plot(workers_list, speedup_amdahl, marker='s', linestyle='--', label=f'Hukum Amdahl (P={P})', color='red')
    plt.plot(workers_list, workers_list, linestyle=':', label='Linear Speedup (Ideal)', color='gray')
    
    plt.xlabel('Jumlah Worker (N)')
    plt.ylabel('Speedup')
    plt.title('Analisis Speedup Skala vs Hukum Amdahl')
    plt.xticks(workers_list)
    plt.legend()
    plt.grid(True)
    plt.show()

Menjalankan dengan 1 worker...


## Nomor 1.
### 1. Analisis Masalah Kenapa Kode Macet di Windows (VSCode)
Keterbatasan Interaktif (VS Code & Jupyter/Interactive Window):
Ketika kode dijalankan secara interaktif di VS Code (baik lewat Jupyter Notebook .ipynb maupun Python Interactive Window), fungsi-fungsi yang didefinisikan secara lokal di dalam sel sering kali gagal melalui proses pickling (serialisasi objek) saat hendak dikirim ke sub-process Windows. Sub-process menunggu data yang tidak pernah sampai, sementara proses utama menunggu worker menyelesaikan tugasnya, menyebabkan program macet tanpa memunculkan pesan error standar.

### 2. Bentuk Keluaran dan Analisis Kurva yang Seharusnya
Jika program ini dijalankan pada lingkungan yang kompatibel (seperti Linux atau Google Colab), keluaran dan analisis yang diperoleh adalah sebagai berikut:
#### a. Keluaran Waktu Eksekusi (Terminal / Output):
    1 Worker: Membutuhkan waktu paling lama karena seluruh beban kerja dijalankan secara sekuensial (berurutan) oleh satu inti CPU.
    2, 4, dan 8 Worker: Waktu eksekusi akan semakin cepat (speedup) seiring bertambahnya jumlah worker karena beban pencarian bilangan prima dibagi rata ke beberapa inti prosesor secara paralel.

#### b. Bentuk Kurva Speedup vs Hukum Amdahl:
##### a. Linear Speedup (Ideal): Garis lurus diagonal di mana penambahan worker berbanding lurus dengan peningkatan kecepatan (misal: 4 worker = 4 kali lebih cepat).
##### b. Kurva Aktual vs Hukum Amdahl: Kurva aktual biasanya akan di bawah kurva teoretis Hukum Amdahl. Peningkatan kecepatan akan melandai (plateau) dan tidak naik secara linear lagi setelah jumlah worker melebihi jumlah core fisik prosesor komputer, akibat adanya overhead komunikasi antar-proses dan prosesor yang harus bergantian (context switching).

In [ ]:
# 2.
import time
import random
import multiprocessing as mp

def worker_queue(queue, jumlah_angka):
    # Setiap proses menjumlahkan 1 juta angka acak
    subtotal = sum(random.randint(1, 100) for _ in range(jumlah_angka))
    # Kirim subtotal ke main process lewat queue
    queue.put(subtotal)

if __name__ == '__main__':
    jumlah_proses = 4
    jumlah_angka_per_proses = 1_000_000
    
    queue = mp.Queue()
    prospek_list = []
    
    start_time = time.time()
    
    # Membuat dan menjalankan 4 proses
    for _ in range(jumlah_proses):
        p = mp.Process(target=worker_queue, args=(queue, jumlah_angka_per_proses))
        prospek_list.append(p)
        p.start()
        
    # Mengambil hasil subtotal dari queue dan menjumlahkannya di main process
    total_keseluruhan = 0
    for _ in range(jumlah_proses):
        total_keseluruhan += queue.get()
        
    # Menunggu semua proses selesai
    for p in prospek_list:
        p.join()
        
    end_time = time.time()
    durasi_queue = end_time - start_time
    
    print(f"[Queue] Total Hasil: {total_keseluruhan}")
    print(f"[Queue] Waktu Eksekusi: {durasi_queue}:.4f detik")

## Nomor 2.
#### 1. Analisis Perbandingan
##### a. Versi Shared Memory (Value) umumnya mencatatkan waktu eksekusi yang sedikit lebih cepat. Hal ini karena proses langsung menuliskan data ke area memori bersama tanpa perlu melakukan serialisasi/deserialisasi pesan yang rumit.
#### b. Versi Message Passing (Queue) sedikit lebih lambat karena membutuhkan proses enkapsulasi data ke dalam antrean (queue) yang dikelola oleh sistem operasi, lalu dikirimkan dari proses anak kembali ke main process.


#### 2. Aspek Keamanan Data & Sinkronisasi (Race Condition):
##### a. Pada Queue, sinkronisasi ditangani secara otomatis dan aman oleh struktur data antrean internal, sehingga risiko data rusak (race condition) sangat kecil karena setiap proses bekerja secara terisolasi sebelum mengirim data.
##### b. Pada Value, penggunaan objek Lock menjadi kunci mutlak. Tanpa adanya Lock, beberapa proses yang mencoba mengubah nilai variabel bersamaan akan saling menimpa data satu sama lain (race condition), yang berakibat pada hasil total akhir yang salah. Namun, terlalu sering menggunakan Lock juga bisa memicu lock contention (proses saling menunggu giliran membuka kunci), yang dapat memperlambat kinerja jika tidak dikelola dengan baik.

### Mengenai Kode yang tidak bisa di run
##### Di Windows, mekanisme spawn mengharuskan Python meluncurkan sub-process baru dari awal dan melakukan pickling terhadap fungsi targetnya. Sering kali, VSCode gagal menangani stream input/output antarproses tersebut, sehingga program terjebak dalam deadlock permanen tanpa memunculkan error. Karena itu tidak bisa di run atau tidak berhenti berhenti prosesnya saat dijalankan.

In [ ]:
import time
import random
import multiprocessing as mp
import sys

def cek_prima(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

# 1. FUNGSI PRODUCER (Antar-Proses)
def producer(q, jumlah_consumers):
    print("[Producer] Mulai memproduksi data!")
    for i in range(1, 20): # Menghasilkan angka untuk diproses secara CPU-bound
        # Mengirimkan angka acak besar yang akan diuji primanya
        item = random.randint(100000, 500000)
        q.put(item)
        print(f"[Producer] Menambahkan angka: {item}")
        time.sleep(0.05) 
    
    print("[Producer] Selesai memproduksi semua data.")
    
    for _ in range(jumlah_consumers):
        q.put(None)

# 2. FUNGSI CONSUMER (Antar-Proses)
def consumer(consumer_id, q):
    print(f"  [Consumer-{consumer_id}] Mulai memproses data!")
    while True:
        item = q.get()
        
        # Cek sinyal berhenti (None)
        if item is None:
            print(f"  [Consumer-{consumer_id}] Menerima sinyal 'None', Berhenti.")
            break
        
        hasil = cek_prima(item)
        print(f"  [Consumer-{consumer_id}] Memproses angka {item} -> Apakah Prima? {hasil}")
        time.sleep(0.1) # Simulasi waktu proses komputasi

if __name__ == '__main__':
    # Memastikan kompatibilitas method context (aman untuk Windows & Linux)
    metode_ctx = 'spawn' if sys.platform.startswith('win') else 'fork'
    ctx = mp.get_context(metode_ctx)

    q = ctx.Queue()
    JUMLAH_CONSUMERS = 3

    # 3. MEMBUAT DAN MENJALANKAN PROSES
    p_producer = ctx.Process(target=producer, args=(q, JUMLAH_CONSUMERS), name="Producer-Process")

    p_consumers = []
    for i in range(1, JUMLAH_CONSUMERS + 1):
        p = ctx.Process(target=consumer, args=(i, q), name=f"Consumer-Process-{i}")
        p_consumers.append(p)

    start_time = time.time()

    p_producer.start()
    for p in p_consumers:
        p.start()

    # Tunggu semua proses selesai
    p_producer.join()
    for p in p_consumers:
        p.join()

    end_time = time.time()
    print(f"\n[Main] Semua proses Producer dan Consumer selesai dalam {end_time - start_time:.4f} detik!")


[Main] Semua proses Producer dan Consumer selesai dalam 0.1722 detik!


## Nomor 3.

#### 1. Analisis Aktual Perbandingan Versi Threading vs Multiprocessing (Producer-Consumer)
##### a. Perubahan Struktur Kode:
1. Secara logika arsitektur, pola kode tidak banyak berubah karena kedua versi sama-sama menggunakan konsep antrean (Queue) untuk memasukkan data (put) dan mengambil data (get).
2. Perbedaan struktural terletak pada pustaka yang diimpor (dari threading dan queue menjadi multiprocessing dan mp.Queue), penggunaan objek proses (mp.Process), serta keharusan menambahkan blok pengaman if __name__ == '__main__': yang wajib ada pada program multiprocessing di Python.

##### b. Perbandingan Performa untuk Tugas CPU-bound:
1. Versi Threading (Pertemuan 2): Kinerja akan sangat lambat atau mengalami bottleneck yang parah jika diberikan tugas komputasi berat (CPU-bound seperti pengecekan bilangan prima). Hal ini disebabkan oleh adanya aturan GIL (Global Interpreter Lock) pada Python, di mana dalam satu waktu, interpreter hanya mengizinkan satu thread yang mengeksekusi bytecode Python murni pada satu inti CPU fisik. Akibatnya, meskipun dibuat banyak thread, beban kerja tetap bergantian dikerjakan pada satu core saja.
2. Versi Multiprocessing: Karena setiap proses memiliki interpreter dan ruang memori yang independen secara fisik, pembatasan GIL berhasil dilewati sepenuhnya (true parallelism). Program dapat mendistribusikan beban kerja komputasi berat ke berbagai inti (core) CPU secara simultan. Meskipun ada sedikit waktu tambahan untuk overhead pembuatan proses (spawn overhead), skalabilitas eksekusi untuk tugas CPU-bound jauh lebih optimal dan efisien dibandingkan menggunakan thread.

#### 2. Analisa tentang output
1. Ketika kode multiprocessing dijalankan di lingkungan lokal Windows (seperti di VS Code), program mengeksekusi metode spawn sebagai pengganti metode fork. Pada mekanisme spawn, setiap proses anak (child process) diluncurkan sebagai interpreter Python yang benar-benar terisolasi dengan ruang memori yang terpisah dari proses utama (main process).

##### Penyebab log dari fungsi producer dan consumer tidak muncul di terminal utama meliputi:
1. Buffer Standar Input/Output (stdout): Pada banyak lingkungan eksekusi terminal di Windows, output yang dihasilkan oleh proses anak (child process) masuk ke dalam penampungan memori (buffer) sistem operasi dan tidak langsung dialirkan (flushed) ke layar terminal utama kecuali proses tersebut selesai atau buffer penuh.
2. Keterlambatan / Kegagalan Aliran Stream: Karena proses berjalan sangat cepat (hanya sepersekian detik), program utama sudah keburu selesai dan menutup seluruh saluran komunikasi sebelum proses anak sempat mencetak teks ke layar konsol terminal interaktif.

## Nomor 4.
### 1. Kapan memilih Manager():   
Dipilih untuk skenario kompleks yang membutuhkan struktur data fleksibel, seperti dictionary atau list dinamis (misalnya: aplikasi web scraper multiproses yang mengumpulkan metadata produk secara bersamaan ke dalam satu dict terpusat).   Trade-off: Menawarkan kemudahan pengembangan karena mendukung Python object native, tetapi memiliki performa lebih lambat (overhead komunikasi antar-proses via IPC/server-process).
### 2. Kapan memilih Value / Array:   
Dipilih untuk skenario high-performance computational dengan tipe data primitif dan ukuran tetap (misalnya: pemrosesan matriks gambar/piksel atau perhitungan numerik skala besar secara paralel).   Trade-off: Menawarkan performa & kecepatan tinggi (karena berbagi shared memory langsung di RAM), namun kurang fleksibel karena tipe data terbatas dan ukurannya fixed/kaku.   